# Rotated surface-code memory with a data-qubit defect

This notebook measures and disables the center data qubit, removes it from neighboring check supports, and alternates the affected Z and X gauge checks. Unaffected checks remain active in every round.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "lightstim").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pymatching

from lightstim.noise.config import NoiseConfig
from lightstim.protocols.rotated_surface_defect import (
    RotatedSurfaceDefectMemoryExperiment,
)

## Build a compact protocol

The production default uses `d` pre-defect rounds and `d` alternating post-defect rounds. This compact circuit keeps one pre-defect round and two post-defect rounds so the complete defect transition is easy to inspect.

In [ ]:
experiment = RotatedSurfaceDefectMemoryExperiment(
    distance=3,
    pre_defect_rounds=1,
    post_defect_schedule=("Z", "X"),
)
clean_circuit = experiment.build()

affected_checks = [
    {
        "uid": uid,
        "basis": experiment.system.stabilizers[uid]["type"],
        "syndrome_coord": experiment.system.stabilizers[uid]["syn_coord"],
        "effective_weight": len(
            experiment.system.effective_stabilizer(uid)["data_indices"]
        ),
    }
    for uid in sorted(experiment.defect.affected_stabilizers)
]
print("defect coordinate:", experiment.defect_coord)
print("gauge schedule:", experiment.gauge_schedule)
print("affected checks:", affected_checks)
print(
    f"qubits={clean_circuit.num_qubits}, "
    f"detectors={clean_circuit.num_detectors}, "
    f"observables={clean_circuit.num_observables}"
)

## Defect transition and gauge rounds

The detector slices begin at the center-qubit measurement and include both post-defect gauge rounds plus final readout.

In [ ]:
tick = 0
defect_tick = None
for instruction in clean_circuit.flattened():
    if instruction.name == "TICK":
        tick += 1
    elif (
        instruction.name in {"M", "MX"}
        and any(
            target.is_qubit_target
            and target.value == experiment.defect_qubit
            for target in instruction.targets_copy()
        )
    ):
        defect_tick = tick
        break

assert defect_tick is not None
clean_circuit.diagram(
    "detslice-with-ops-svg",
    tick=range(defect_tick, clean_circuit.num_ticks + 1),
)

## Noisy PyMatching smoke run

The decomposed detector error model remains graph-like and can be compiled by PyMatching. Use `benchmarks/memory/run_memory.py --codes rotated_sc_defect` for checkpointed LER sweeps.

In [ ]:
p = 1e-3
noise = NoiseConfig(
    p_idle=p, p_1q=p, p_2q=p, p_meas=p, p_reset=p
)
noisy_circuit = RotatedSurfaceDefectMemoryExperiment(
    distance=3,
    noise_params=noise,
).build()
dem = noisy_circuit.detector_error_model(decompose_errors=True)
matching = pymatching.Matching.from_detector_error_model(dem)
detections, observables = noisy_circuit.compile_detector_sampler().sample(
    2_000, separate_observables=True
)
predictions = matching.decode_batch(detections)
logical_error_rate = np.mean(predictions != observables)

print("shortest graph-like error:", len(noisy_circuit.shortest_graphlike_error()))
print("pilot logical error rate:", logical_error_rate)